# Gemma 12B JAX 90 Baseline

Run 90 reviewed diagnosis pairs, one prompt per case in this arm, with the previous 12B settings. The four-L4 runtime uses checkpoint v2, wheelhouse v31 and the pinned competition image. Download gemma-12b-hints-90-baseline.zip when finished.

In [ ]:
RUN_INFERENCE = True
ALLOCATION_START_UTC_EPOCH = None


In [ ]:
import base64, hashlib, os, subprocess, sys, tempfile
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
stage = Path(tempfile.mkdtemp(prefix="gemma-full-"))
wheel = stage / "jax_diagnosis_paper-0.6.0-py3-none-any.whl"
payload = base64.b64decode("...", validate=True)
assert hashlib.sha256(payload).hexdigest() == "a063dd8f53473ce052c6d8c8401c150845dfc24b3e570c50912372c352ec2c6b"
wheel.write_bytes(payload)
installed = stage / "installed"
subprocess.run([sys.executable,"-m","pip","install","--no-index","--no-deps","--no-compile","--target",str(installed),str(wheel)],check=True)
environment = os.environ.copy()
environment["PYTHONPATH"] = str(installed) + os.pathsep + environment.get("PYTHONPATH","")
command = [sys.executable,"-B","-m","jax_diagnosis_paper.full_launch",'baseline']
subprocess.run(command,env=environment,check=True)


In [ ]:
if RUN_INFERENCE:
    result = Path("/kaggle/working") / "gemma-12b-hints-90-baseline"
    archive = result.with_suffix(".zip")
    if result.exists() or archive.exists():
        raise RuntimeError("Existing run artifacts. Preserve the completed or partial run.")
    args = ["--run"]
    if ALLOCATION_START_UTC_EPOCH is not None: args += ["--allocation-start",str(ALLOCATION_START_UTC_EPOCH)]
    try:
        subprocess.run(command + args,env=environment,check=True)
    finally:
        if result.is_dir():
            with ZipFile(archive,"x",ZIP_DEFLATED) as z:
                for file in sorted(result.iterdir()):
                    if file.is_file(): z.write(file,file.name)
else:
    print("Preflight complete. No generation requested.")
